# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SaimAli0001/Flyrank-Internship-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Answer

**Unit of analysis**

The warehouse stores one daily observation for a single content page (`content_hash_id`) belonging to a single client (`client_hash_id`). Each row represents the search performance of that page on one specific `report_date`.

For this project, these daily observations from March 2026 will later be aggregated into one row per content page to create the feature frame used for ranking refresh candidates.

**Time window**

The feature window is March 2026. This month was selected because it is a middle month in the warehouse, allowing feature engineering without using the final month as development data.

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
%pip install python-dotenv
%pip install duckdb

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [22]:
from dotenv import load_dotenv
import os

load_dotenv("../../.env")

token = os.getenv("HF_TOKEN")

print("Token loaded:", token is not None)

Token loaded: True


In [23]:
import duckdb

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN '{token}'
)
""")

print("DuckDB secret created")

DuckDB secret created


In [24]:
REL = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

In [25]:
schema = con.sql(f"""
DESCRIBE SELECT * FROM {REL}
""").df()

schema

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [26]:
CONTENT_REL = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
)
"""

content_schema = con.sql(f"""
    DESCRIBE SELECT * FROM {CONTENT_REL}
""").df()

content_schema

,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,keyword_hash_id,VARCHAR,YES,None,None,None
3,url_hash_id,VARCHAR,YES,None,None,None
4,keyword_char_count,BIGINT,YES,None,None,None
5,keyword_token_count,BIGINT,YES,None,None,None
6,url_char_count,BIGINT,YES,None,None,None
7,content_created_date,DATE,YES,None,None,None
8,content_updated_date,DATE,YES,None,None,None
9,content_type,VARCHAR,YES,None,None,None


### Observation

The warehouse grain is one content page for one client on one reporting day. For feature engineering, the March daily observations will later be aggregated into one row per content page, matching the decision of selecting pages for content refresh.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features

The following fields are used as model features because they are observable search signals that are available at the end of March 2026.

- March impressions
- March CTR
- March average position
- March impression trend
- March CTR trend

### Label (Proxy)

The objective of this project is to rank pages that should be reviewed first for content refresh.

Since the warehouse does not contain a verified post-refresh outcome, a true supervised label is not available. Therefore, I define a proxy label called **Refresh Priority**, which identifies pages with high visibility but relatively low click-through performance within the same client. This proxy supports decision-making rather than claiming that refreshing the page will guarantee better results.

### Context

The following fields identify each observation or provide grouping information, but they are not used as predictive features.

- `client_hash_id`
- `content_hash_id`
- `report_date`
- `month`

### Excluded

The following fields are intentionally excluded from modelling.

- GA4 engagement metrics, because coverage is limited across March pages and many pages contain insufficient GA4 history.
- Future information (April onwards), because it would introduce data leakage.
- Any fields that directly encode or reconstruct the target, because they would leak the answer to the model.

### Observation

The selected features are observable signals that were available before the decision point. Context fields are used only for identification and grouping, while excluded fields are omitted either because they have insufficient coverage or because they could introduce data leakage.

In [33]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Verification Queries

The following SQL queries verify the assumptions made in the data contract.

1. Verify the unit of analysis (grain).
2. Verify that the analysis uses only the March 2026 feature window.
3. Verify that the required search-performance data is available for feature engineering.

These checks ensure that the selected data matches the intended decision point before feature creation begins.

In [ ]:
grain_check = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT content_hash_id) AS unique_pages,
    COUNT(DISTINCT report_date) AS unique_days
FROM {REL}
""").df()

grain_check

,total_rows,unique_pages,unique_days
0,9841378,331437,31


In [ ]:
ga4_check = con.sql("""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS ga4_available_rows,

        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
              AND ga4_sessions > 0
        ) AS rows_with_sessions

    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
""").df()

ga4_check

,total_rows,ga4_available_rows,rows_with_sessions
0,9841378,413966,410335


In [ ]:
page_ga4_check = con.sql("""
    SELECT
        COUNT(DISTINCT content_hash_id) AS total_pages,

        COUNT(DISTINCT CASE
            WHEN ga4_data_available IS TRUE
            THEN content_hash_id
        END) AS pages_with_ga4,

        COUNT(DISTINCT CASE
            WHEN ga4_data_available IS TRUE
             AND ga4_sessions > 0
            THEN content_hash_id
        END) AS pages_with_sessions

    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
""").df()

page_ga4_check

,total_pages,pages_with_ga4,pages_with_sessions
0,331437,90489,90237


In [ ]:
ga4_days = con.sql("""
    SELECT
        content_hash_id,
        COUNT(DISTINCT report_date) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS ga4_days

    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )

    GROUP BY content_hash_id
""").df()

ga4_days[ga4_days["ga4_days"] > 0]["ga4_days"].describe()

count    90489.000000
mean         4.574766
std          5.682016
min          1.000000
25%          1.000000
50%          2.000000
75%          6.000000
max         31.000000
Name: ga4_days, dtype: float64

In [ ]:
search_volume_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_pages,
        COUNT(search_volume) AS pages_with_search_volume,
        COUNT(*) - COUNT(search_volume) AS pages_missing_search_volume
    FROM {CONTENT_REL}
""").df()

search_volume_check

,total_pages,pages_with_search_volume,pages_missing_search_volume
0,519606,376984,142622


### Observation

This query confirms that each unique `content_hash_id` represents one content page in the content dimension. This supports using one row per content page when building the feature frame.

In [36]:
march_check = con.sql(f"""
SELECT
    MIN(report_date) AS first_day,
    MAX(report_date) AS last_day,
    COUNT(*) AS total_rows
FROM {REL}
WHERE month='2026-03'
""").df()

march_check

,first_day,last_day,total_rows
0,2026-03-01,2026-03-31,9841378


### Observation

The query confirms that the selected feature window contains only March 2026 data. All features will therefore be created using information available before the decision point.

In [37]:
availability_check = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS rows_after_filter
FROM {REL}
""").df()

availability_check

,total_rows,rows_after_filter
0,9841378,3611061


### Observation

The query verifies that Google Search Console data is available for the selected rows. Only observations where `gsc_data_available IS TRUE` will be used when calculating impressions, CTR and average position.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data Limits

This analysis has several limitations.

1. The feature window is limited to March 2026, so it cannot capture longer-term trends outside this month.

2. The warehouse does not contain a verified post-refresh outcome, so the project uses a proxy label (Refresh Priority) instead of a true supervised target.

3. GA4 coverage is limited across March pages, therefore engagement metrics cannot be used consistently for every content page for many pages, so engagement metrics were excluded from the feature set.

4. The model ranks pages for review based on observed search performance. It cannot determine whether refreshing a page will definitely improve its future performance.

In [38]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
data_limit_check = con.sql(f"""
SELECT
    COUNT(DISTINCT content_hash_id) AS total_pages,
    COUNT(DISTINCT CASE
        WHEN ga4_data_available IS TRUE
        THEN content_hash_id
    END) AS pages_with_ga4
FROM {REL}
""").df()

data_limit_check

,total_pages,pages_with_ga4
0,331437,90489


### Observation

The query shows that only a subset of pages contains GA4 data. Because engagement information is unavailable for many pages, the final feature set relies primarily on Google Search Console metrics.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.